# Car Features and MSRP — Exploratory Data Analysis
**Dataset:** [Car Features and MSRP — Kaggle](https://www.kaggle.com/datasets/CooperUnion/cardataset)  
**Student:** Replace with your name | **Course:** EDA Assignment 09

> Download `data.csv` from Kaggle and place it in the same folder as this notebook. Run every cell in order. Read and personalize the explanations after seeing your actual outputs.

## Step 1 — Understand the data
One row represents one vehicle configuration/listing. The columns describe manufacturer, model, year, engine, transmission, drivetrain, body style, fuel economy, popularity and MSRP.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style="whitegrid")
DATA_PATH = Path("data.csv")
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("Data type"))
display(df.describe(include="all").T)
display(pd.DataFrame({"Column": df.columns, "Meaning": [
    "Car manufacturer", "Car model", "Model year", "Fuel type", "Engine horsepower",
    "Number of engine cylinders", "Transmission category", "Driven wheels",
    "Number of doors", "Market segment/category", "Vehicle size", "Vehicle body style",
    "Highway fuel economy (MPG)", "City fuel economy (MPG)", "Popularity index",
    "Manufacturer suggested retail price"
]}))

## Step 2 — Research questions
1. Which manufacturers have the most listings?
2. What is the distribution of MSRP?
3. Which vehicle styles are most common?
4. How does engine horsepower relate to MSRP?
5. How does engine size (cylinders) relate to MSRP?
6. Does transmission type differ in typical MSRP?
7. Which vehicle sizes have the highest median MSRP?
8. How does city fuel economy vary with highway fuel economy?

## Step 3 — Data cleaning
We inspect missing values, duplicates, data types and outliers. Decisions below are explicit; review the results before final submission.

In [ ]:
print("Missing values before cleaning:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing count"))
df.isna().sum().sort_values(ascending=False).plot(kind="bar", figsize=(12,4), title="Missing Values by Column")
plt.ylabel("Missing values"); plt.xlabel("Column"); plt.tight_layout(); plt.show()

print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()

# Normalize column names for reliable coding
df.columns = [c.strip().lower().replace(" ", "_").replace(".", "_") for c in df.columns]
numeric_cols = ["year","engine_hp","engine_cylinders","number_of_doors","highway_mpg","city_mpg","popularity","msrp"]
for col in numeric_cols:
    if col in df.columns: df[col] = pd.to_numeric(df[col], errors="coerce")
print(df.dtypes)

# Fill numeric missing values with median; categorical missing values with "Unknown".
# This retains records while avoiding mean sensitivity to skewed distributions.
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(exclude=np.number).columns:
    df[col] = df[col].fillna("Unknown")

# IQR outlier check for at least two numeric columns; keep values because expensive/high-performance
# vehicles can be legitimate observations. Flagging is not the same as deleting.
for col in ["msrp","engine_hp"]:
    q1, q3 = df[col].quantile([.25,.75]); iqr=q3-q1
    lo, hi = q1-1.5*iqr, q3+1.5*iqr
    print(col, "IQR lower:", lo, "upper:", hi, "outliers:", ((df[col]<lo)|(df[col]>hi)).sum())
    plt.figure(figsize=(7,3)); sns.boxplot(x=df[col]); plt.title(f"{col} — IQR outlier check"); plt.tight_layout(); plt.show()

print("Cleaned shape:", df.shape)
display(df.isna().sum().to_frame("Missing after cleaning"))

## Step 4 — Univariate analysis

In [ ]:
for col in ["msrp","engine_hp","year","city_mpg","highway_mpg"]:
    plt.figure(figsize=(7,4)); sns.histplot(df[col], kde=True); plt.title(f"Distribution of {col}"); plt.xlabel(col); plt.ylabel("Count"); plt.tight_layout(); plt.show()
for col in ["make","transmission_type","vehicle_size","vehicle_style","engine_fuel_type"]:
    if col in df:
        plt.figure(figsize=(10,4)); df[col].value_counts().head(12).plot(kind="bar"); plt.title(f"Top categories: {col}"); plt.xlabel(col); plt.ylabel("Count"); plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()

## Step 5 — Bivariate analysis

In [ ]:
plt.figure(figsize=(7,5)); sns.scatterplot(data=df.sample(min(3000,len(df)),random_state=42),x="engine_hp",y="msrp",alpha=.5); plt.title("Engine horsepower vs MSRP"); plt.tight_layout(); plt.show()
plt.figure(figsize=(9,6)); sns.heatmap(df.select_dtypes(include=np.number).corr(),annot=True,cmap="coolwarm",fmt=".2f"); plt.title("Numeric feature correlation"); plt.tight_layout(); plt.show()
plt.figure(figsize=(9,5)); sns.boxplot(data=df,x="transmission_type",y="msrp"); plt.title("MSRP by transmission type"); plt.xticks(rotation=20); plt.tight_layout(); plt.show()
plt.figure(figsize=(8,5)); sns.boxplot(data=df,x="vehicle_size",y="msrp"); plt.title("MSRP by vehicle size"); plt.tight_layout(); plt.show()

## Step 6 — Answer the research questions
Run the cell below to produce eight visual answers. Add 1–2 sentences under each chart in your submitted notebook, using the actual chart values.

In [ ]:
fig, ax = plt.subplots(4,2,figsize=(16,22))
df["make"].value_counts().head(10).plot(kind="bar",ax=ax[0,0],title="Q1. Top manufacturers")
sns.histplot(df["msrp"],ax=ax[0,1]); ax[0,1].set_title("Q2. MSRP distribution")
df["vehicle_style"].value_counts().head(10).plot(kind="bar",ax=ax[1,0],title="Q3. Common vehicle styles")
sns.scatterplot(data=df.sample(min(2500,len(df)),random_state=42),x="engine_hp",y="msrp",ax=ax[1,1],alpha=.5); ax[1,1].set_title("Q4. Horsepower vs MSRP")
sns.boxplot(data=df,x="engine_cylinders",y="msrp",ax=ax[2,0]); ax[2,0].set_title("Q5. Cylinders vs MSRP")
sns.boxplot(data=df,x="transmission_type",y="msrp",ax=ax[2,1]); ax[2,1].set_title("Q6. MSRP by transmission")
sns.boxplot(data=df,x="vehicle_size",y="msrp",ax=ax[3,0]); ax[3,0].set_title("Q7. MSRP by vehicle size")
sns.scatterplot(data=df.sample(min(2500,len(df)),random_state=42),x="city_mpg",y="highway_mpg",ax=ax[3,1],alpha=.5); ax[3,1].set_title("Q8. City vs highway MPG")
for a in ax.flat: a.tick_params(axis="x",rotation=35)
plt.tight_layout(); plt.show()

## Step 7 — Conclusion
Write three findings supported by your charts and one practical suggestion. Avoid inventing percentages; calculate them from your cleaned dataset.

In [ ]:
print("Top manufacturer:", df["make"].value_counts().idxmax())
print("Median MSRP:", round(df["msrp"].median(),2))
print("Median horsepower:", round(df["engine_hp"].median(),2))
print("Correlation horsepower/MSRP:", round(df[["engine_hp","msrp"]].corr().iloc[0,1],3))